# Link prediction with LPFormer (Cora)

LPFormer ([Shomer et al., 2024](https://arxiv.org/abs/2310.11009)) scores a candidate link with a
graph transformer: besides the GCN embeddings of the two nodes, it attends over other nodes and
counts the structure around the pair (common neighbors, one-hop neighbors). Note that K3-Node's
`LPFormer` is a simplified version of PyG's: it does not yet use personalized PageRank to choose
and weight the nodes it attends to.

Same model as PyG's [`examples/lpformer.py`](https://github.com/pyg-team/pytorch_geometric/blob/master/examples/lpformer.py); on Cora instead of ogbl-ppa, with AUC instead of Hits@100.

In [ ]:
# Install K3-Node, then choose a backend: "tensorflow", "torch" or "jax"
!pip install -q "k3-node[examples] @ git+https://github.com/anas-rz/k3-node"
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Load the data

`RandomLinkSplit` hides 5% of the edges for validation and 10% for testing. Every split keeps the edges used for message passing in `edge_index`; the edges to predict are in `edge_label_index` with labels `edge_label` (1 for a true edge, 0 for a sampled non-edge).

In [ ]:
import keras
from keras import ops
from k3_node.datasets import Planetoid
from k3_node.loader import FullGraphDataset
from k3_node.models import LPFormer
from k3_node.transforms import Compose, NormalizeFeatures, RandomLinkSplit

transform = Compose([
    NormalizeFeatures(),
    RandomLinkSplit(num_val=0.05, num_test=0.1, is_undirected=True, add_negative_train_samples=False),
])
dataset = Planetoid("data/Planetoid", name="Cora", transform=transform)
train_data, val_data, test_data = dataset[0]
print(train_data)

## Define the model

The structure counts are computed on the host, so the model runs eagerly (`run_eagerly=True` below).

In [ ]:
class LinkModel(keras.Model):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.lpformer = LPFormer(in_channels, hidden_channels, num_gnn_layers=3, gnn_dropout=0.1,
                                 transformer_dropout=0.1)

    def call(self, data, training=False):
        out = self.lpformer(data.edge_label_index, data.x, data.edge_index, training=training)
        return ops.squeeze(out, axis=-1)


model = LinkModel(dataset.num_features, 64)

## Train

`neg_sampling_ratio=1.0` adds as many random non-edges as there are training edges, sampled anew every epoch. The score of an edge is a logit, so the loss is binary cross-entropy and the metric is the area under the ROC curve (AUC).

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.AUC(from_logits=True, name="auc")],
    run_eagerly=True,
)
model.fit(
    FullGraphDataset(train_data, neg_sampling_ratio=1.0),
    validation_data=FullGraphDataset(val_data),
    epochs=100,
    verbose=2,
)

## Evaluate

In [ ]:
loss, auc = model.evaluate(FullGraphDataset(test_data), verbose=0)
print(f"Test AUC: {auc:.4f}")